# 04 - Ensemble (GB + Pretrained)

Mục tiêu: kết hợp dự báo từ GB baseline (02_baseline_gb.ipynb) và
pretrained model (03_pretrained.ipynb) bằng simple average và
weighted average, sau đó đánh giá xem ensemble có cải thiện so với
từng model riêng lẻ hay không.

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from evaluation import calculate_metrics
from ensemble import simple_average, tune_weight, weighted_average

## Bước 1 — Load và kiểm tra 2 file predictions đã khớp nhau

In [2]:
gb_pred = pd.read_csv("../outputs/metrics/gb_baseline_predictions.csv")
pretrained_pred = pd.read_csv("../outputs/metrics/pretrained_predictions.csv")

print("GB predictions:", gb_pred.shape, gb_pred.columns.tolist())
print("Pretrained predictions:", pretrained_pred.shape, pretrained_pred.columns.tolist())

# Kiểm tra 2 tập series có khớp nhau không
gb_ids = set(gb_pred["unique_id"])
pretrained_ids = set(pretrained_pred["unique_id"])
print("Series chỉ có ở GB:", len(gb_ids - pretrained_ids))
print("Series chỉ có ở Pretrained:", len(pretrained_ids - gb_ids))

# Kiểm tra khoảng thời gian 2 file có trùng nhau không (bắt buộc để merge đúng)
print("\nKhoảng ds trong gb_pred:", gb_pred["ds"].min(), "->", gb_pred["ds"].max())
print("Khoảng ds trong pretrained_pred:", pretrained_pred["ds"].min(), "->", pretrained_pred["ds"].max())

GB predictions: (8334, 7) ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_lgb', 'pred_xgb']
Pretrained predictions: (8334, 6) ['unique_id', 'length_group', 'category', 'ds', 'y', 'pred_pretrained']
Series chỉ có ở GB: 0
Series chỉ có ở Pretrained: 0

Khoảng ds trong gb_pred: 1902-02-01 -> 1932-07-01
Khoảng ds trong pretrained_pred: 1902-02-01 -> 1932-07-01


## Bước 2 — Merge 2 nguồn dự báo theo (unique_id, ds)

In [3]:
merged = gb_pred.merge(
    pretrained_pred[["unique_id", "ds", "pred_pretrained"]],
    on=["unique_id", "ds"],
    how="inner"
)

print(f"Số dòng sau merge: {len(merged)}")
assert len(merged) > 0, "Merge rỗng — kiểm tra lại khoảng thời gian giữa 2 file"

Số dòng sau merge: 8334


## Bước 3 — Xác định model GB đại diện

Chọn model tốt hơn giữa LightGBM/XGBoost làm đại diện GB baseline
cho phần ensemble, tránh làm loãng thí nghiệm với 2 model GB cùng lúc.

In [4]:
metrics_lgb = calculate_metrics(merged["y"], merged["pred_lgb"])
metrics_xgb = calculate_metrics(merged["y"], merged["pred_xgb"])

best_gb_col = "pred_lgb" if metrics_lgb["RMSE"] < metrics_xgb["RMSE"] else "pred_xgb"
print(f"LightGBM RMSE: {metrics_lgb['RMSE']:.2f}")
print(f"XGBoost RMSE: {metrics_xgb['RMSE']:.2f}")
print(f"Model GB đại diện: {best_gb_col}")

merged["pred_gb"] = merged[best_gb_col]

LightGBM RMSE: 1237.46
XGBoost RMSE: 1284.56
Model GB đại diện: pred_lgb


## Bước 4 — Tạo các phương án Ensemble (Simple Average & Weighted Average)

Logic ensemble được định nghĩa trong `src/ensemble.py`

Trọng số weighted average được tune trên `tune_set` (70% series),
đánh giá cuối cùng trên `eval_set` (30% series còn lại) — tách theo
từng series để tránh rò rỉ thông tin, và tránh tune/đánh giá trên
cùng 1 tập dữ liệu.

In [5]:
# Simple average
merged["pred_ensemble_avg"] = simple_average(merged["pred_gb"], merged["pred_pretrained"])

# Tách tune/eval theo series (không theo dòng)
unique_ids = merged["unique_id"].unique()
tune_ids, eval_ids = train_test_split(unique_ids, test_size=0.3, random_state=42)
tune_set = merged[merged["unique_id"].isin(tune_ids)]

# Tune trọng số weighted average trên tune_set
best_w, best_rmse = tune_weight(tune_set)
print(f"Trọng số tối ưu: w={best_w:.2f} (RMSE trên tune set: {best_rmse:.2f})")

# Áp dụng trọng số đã tune cho toàn bộ merged
merged["pred_ensemble_weighted"] = weighted_average(
    merged["pred_gb"], merged["pred_pretrained"], best_w
)

# Cắt eval_set SAU CÙNG, khi merged đã có đủ mọi cột dự báo
eval_set = merged[merged["unique_id"].isin(eval_ids)]

Trọng số tối ưu: w=0.85 (RMSE trên tune set: 1359.68)


## Bước 5 — So sánh các phương pháp trên eval_set (chưa dùng để tune)

In [6]:
methods = {
    "GB only": "pred_gb",
    "Pretrained only": "pred_pretrained",
    "Ensemble (avg)": "pred_ensemble_avg",
    "Ensemble (weighted)": "pred_ensemble_weighted",
}

print("Kết quả trên eval_set (không dùng để tune weight):")
for name, col in methods.items():
    m = calculate_metrics(eval_set["y"], eval_set[col])
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}")

Kết quả trên eval_set (không dùng để tune weight):
GB only: RMSE=853.77, MAE=400.53
Pretrained only: RMSE=1034.77, MAE=544.64
Ensemble (avg): RMSE=835.52, MAE=440.98
Ensemble (weighted): RMSE=822.99, MAE=402.20


## Bước 6 — Lưu kết quả đầy đủ, kèm length_group cho bước phân tích chính

In [7]:
final_results = merged[[
    "unique_id", "length_group", "category", "ds", "y",
    "pred_gb", "pred_pretrained", "pred_ensemble_avg", "pred_ensemble_weighted"
]].copy()

final_results.to_csv("../outputs/metrics/ensemble_predictions.csv", index=False)
print(f"Đã lưu {len(final_results)} dòng vào ensemble_predictions.csv")

Đã lưu 8334 dòng vào ensemble_predictions.csv


## Kết luận

Đã lưu `ensemble_predictions.csv` với đầy đủ dự báo từ GB, Pretrained,
Ensemble (avg/weighted), kèm `length_group` và `category` — sẵn sàng
cho `05_gain_analysis.ipynb` để trả lời câu hỏi nghiên cứu chính:
liệu lợi ích của ensemble có phụ thuộc vào độ dài lịch sử dữ liệu
hay không.